# 0) Variables



In [1]:
!nvidia-smi

Sun Nov 15 03:27:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# @title Dataset variables

dataset_in_path = 'data/dataset-source' # @param {type:"string"}

dataset_out_path = 'data/dataset-article' # @param {type:"string"}

patch_size = "72"  # @param [40, 48, 56, 64, 72, 80, 88, 96]
patch_size = int(patch_size)

classes = ["stone"]  # @param {type:"raw"}


In [3]:
# @title Training variables

output_result_subdir="results-kidney" # @param {type:"string"}

model_type="ResNet50" # @param ["EfficientNetV2S", "MobileNetV3Large", "ResNet50", "DenseNet121", "ConvNeXtTiny"]

epochs = 32 # @param {type:"integer"}

early_stop_patience = 16 # @param {type:"integer"}

batch_size = 64 # @param {type:"integer"}


# 1) Output directory

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2) Download segmentation dataset

In [7]:
#@title Load the file ~/.kaggle/kaggle.json
#@markdown Load the kaggle config file

from google.colab import files
import os
import shutil

# 1. Selecionar o kaggle.json
uploaded = files.upload()

# 2. Garantir que o arquivo correto foi enviado
if "kaggle.json" not in uploaded:
    raise RuntimeError("Por favor, selecione o arquivo kaggle.json")

# 3. Criar o diretório ~/.kaggle
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

# 4. Mover o arquivo para o local correto
shutil.move("kaggle.json", os.path.join(kaggle_dir, "kaggle.json"))

# 5. Ajustar permissões
os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)

print("✅ kaggle.json configurado com sucesso em ~/.kaggle/")


Saving kaggle.json to kaggle.json
✅ kaggle.json configurado com sucesso em ~/.kaggle/


In [8]:
!kaggle datasets download -d murillobouzon/kssd2025-kidney-stone-segmentation-dataset -p tmpdat/ --unzip

!mkdir -p {dataset_in_path}/stone
!cp -rf tmpdat/data/image {dataset_in_path}/stone/
!cp -rf tmpdat/data/label {dataset_in_path}/stone/
!rm -rf tmpdat/

Dataset URL: https://www.kaggle.com/datasets/murillobouzon/kssd2025-kidney-stone-segmentation-dataset
License(s): unknown
100% 78.5M/78.5M [00:00<00:00, 86.2MB/s]



# 3) Download project source code

In [9]:
!git clone https://github.com/trucomanx/LocalVsGlobalKidneyStone.git
!pip install -r LocalVsGlobalKidneyStone/1-dataset-processing/requirements.txt

Cloning into 'LocalVsGlobalKidneyStone'...
remote: Enumerating objects: 415, done.
remote: Counting objects: 100% (270/270), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 415 (delta 162), reused 192 (delta 87), pack-reused 145 (from 4)
Receiving objects: 100% (415/415), 3.04 MiB | 22.58 MiB/s, done.
Resolving deltas: 100% (199/199), done.


# 4) Generate classification dataset

In [11]:
import sys
sys.path.append("LocalVsGlobalKidneyStone/1-dataset-processing/src")

from prog_generate import generate_dataset

patch_dataset_path=generate_dataset(dataset_in_path,dataset_out_path,patch_size,classes=classes)


Working patch_size: 72
Generating class: stone

class_name: stone
data/dataset-source/stone/image/*.tif
len: 838


Processando imagens: 100%|██████████| 838/838 [00:41<00:00, 20.09it/s]
Processing data for 1 folds: 113297it [00:04, 27718.35it/s]
Processing data for 2 folds: 113297it [00:02, 39466.47it/s]
Processing data for 3 folds: 113297it [00:02, 40877.85it/s]
Processing data for 4 folds: 113297it [00:03, 37636.11it/s]
Processing data for 5 folds: 113297it [00:03, 30731.83it/s]


In [12]:
import json

fpath=os.path.join(patch_dataset_path,"report.json")
with open(fpath, "r", encoding="utf-8") as f:
    data = json.load(f)
print(data)

{'with-stone': 6408, 'without-stone': 135214}


# 5) Training

In [ ]:
sys.path.append("LocalVsGlobalKidneyStone/2-training-kfold")

from prog_train import train_subdataset
from prog_train import compute_class_weight_from_dict

results_full_dir=os.path.join("drive","MyDrive",output_result_subdir)

dir_path = os.path.join(results_full_dir,model_type,f"dataset-{patch_size}")
train_subdataset( patch_dataset_path,
                  dir_path,
                  model_type,
                  batch_size = batch_size,
                  class_weight = compute_class_weight_from_dict(data),
                  epochs_stage_1 = epochs,
                  epochs_stage_2 = epochs,
                  learning_rate_stage_1 = 1e-3,
                  learning_rate_stage_2 = 1e-4,
                  early_stop_patience = early_stop_patience,
                  my_seed = 42
                 )

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

=== Fold 0 ===
Epoch 1/32
1771/1771 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 0.9114 - loss: 0.2356
Epoch 1: val_loss improved from None to 0.13202, saving model to drive/MyDrive/results-kidney/ResNet50/dataset-72/fold0/best-model-stage1.keras

Epoch 1: finished saving model to drive/MyDrive/results-kidney/ResNet50/dataset-72/fold0/best-model-stage1.keras
1771/1771 ━━━━━━━━━━━━━━━━━━━━ 383s 201ms/step - accuracy: 0.9352 - loss: 0.1651 - val_accuracy: 0.9482 - val_loss: 0.1320
Epoch 2/32
1770/1771 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.9577 - loss: 0.1101
Epoch 2: val_loss improved from 0.13202 to 0.06681, saving model to drive/MyDrive/results-kidney/ResNet50/dataset-72/fold0/best-model-stage1.keras

Epoch 2: finished saving model to drive/MyDrive/results-kidney/ResNet50/dataset-72/fold0/best-model-stage1.keras
1771/1771 ━━━━━━━━━━━━━━━━━━━━ 331s 187ms/step - accuracy: 0.9602 - loss: 0.1067 - val_accuracy: 0.9778 - val_l